# National LMIS — 02: Data Cleaning & Validation Pipeline
**Ministry of Skill Development & Entrepreneurship (MSDE) | SIH_26246**

This notebook validates data integrity, enforces logical domain constraints, and exports clean datasets:
1. Verify `candidates_certified <= candidates_enrolled`.
2. Verify `candidates_placed <= candidates_trained`.
3. Format text keys (trim whitespace, title case).
4. Standardize date formats to `%Y-%m`.


In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path('..').resolve()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


### 1. Clean Job Postings


In [ ]:
df_jobs = pd.read_csv(RAW_DIR / 'job_postings.csv')
df_jobs['job_postings'] = pd.to_numeric(df_jobs['job_postings'], errors='coerce').fillna(0).astype(int)
df_jobs['hiring_growth'] = pd.to_numeric(df_jobs['hiring_growth'], errors='coerce').fillna(0.0)
df_jobs.drop_duplicates(subset=['state', 'district', 'sector', 'trade', 'month'], inplace=True)
df_jobs.to_csv(PROCESSED_DIR / 'cleaned_job_postings.csv', index=False)
print(f'Cleaned Job Postings: {len(df_jobs)} records saved.')


### 2. Clean Training Data with Business Constraints


In [ ]:
train_path = RAW_DIR / 'training_data.csv'
if not train_path.exists():
    train_path = RAW_DIR / 'training_capacity.csv'
df_train = pd.read_csv(train_path)
# Enforce logical rule: certified cannot exceed enrolled
df_train['candidates_certified'] = df_train[['candidates_certified', 'candidates_enrolled']].min(axis=1)
df_train.to_csv(PROCESSED_DIR / 'cleaned_training_data.csv', index=False)
print(f'Cleaned Training Data: {len(df_train)} records saved.')


### 3. Clean Placement Outcomes


In [ ]:
df_place = pd.read_csv(RAW_DIR / 'placements.csv')
# Enforce logical rule: placed cannot exceed trained
df_place['candidates_placed'] = df_place[['candidates_placed', 'candidates_trained']].min(axis=1)
df_place['placement_rate'] = (df_place['candidates_placed'] / df_place['candidates_trained'].replace(0, 1)).clip(0.0, 1.0)
df_place.to_csv(PROCESSED_DIR / 'cleaned_placements.csv', index=False)
print(f'Cleaned Placements: {len(df_place)} records saved.')


### 4. Verification Check


In [ ]:
print('All processed datasets created successfully in data/processed/:')
for f in sorted(list(PROCESSED_DIR.glob('cleaned_*.csv'))):
    print(f' - {f.name} ({f.stat().st_size} bytes)')
